# Tuần 2 - Tổ chức dataset và sinh metadata Tiny-GenImage

Notebook này chuẩn bị metadata cho dataset **Tiny-GenImage** từ Kaggle:

`https://www.kaggle.com/datasets/yangsangtai/tiny-genimage`

Mục tiêu:

1. Kiểm tra cấu trúc dataset.
2. Quét toàn bộ ảnh.
3. Kiểm tra ảnh hợp lệ bằng PIL.
4. Phát hiện ảnh lỗi.
5. Phát hiện ảnh trùng bằng SHA-256.
6. Sinh metadata cho hai trường hợp: cross-generator và combined metadata.
7. Sinh báo cáo `report/week2_dataset_organization_report.md`.

Notebook chỉ chuẩn bị dữ liệu và metadata, không huấn luyện model.

## 0. Cấu hình

Dataset Tiny-GenImage được giữ nguyên split gốc:

```text
Generator/
  train/
    ai/
    nature/
  val/
    ai/
    nature/
```

Quy ước label:

- `nature = real = 0`
- `ai = fake = 1`

`BASE_GENERATOR` dùng cho trường hợp cross-generator. Nếu generator này không tồn tại trong dữ liệu đã tải, notebook sẽ báo lỗi và hiển thị danh sách generator thực tế tìm thấy.

In [ ]:
# Nếu môi trường Colab chưa có kagglehub, chạy dòng dưới đây trước.
# !pip install kagglehub

from pathlib import Path
import hashlib
import json

import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

DATASET_SLUG = "yangsangtai/tiny-genimage"

# Đổi giá trị này theo tên thư mục generator thực tế tìm thấy sau khi tải dataset.
# Với Tiny-GenImage trên Kaggle, ví dụ hợp lệ: "imagenet_ai_0419_biggan".
BASE_GENERATOR = None

WORKING_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = WORKING_ROOT / "data" / "metadata"
REPORT_ROOT = WORKING_ROOT / "report"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_SPLITS = ["train", "val"]
EXPECTED_LABELS = ["ai", "nature"]
LABEL_MAP = {"nature": 0, "ai": 1}
LABEL_DISPLAY = {"nature": "real", "ai": "fake"}

print("WORKING_ROOT:", WORKING_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("REPORT_ROOT:", REPORT_ROOT)
print("BASE_GENERATOR:", BASE_GENERATOR)

## 1. Tải dataset từ Kaggle

Cell này dùng `kagglehub` để tải dataset. Nếu bạn đã có dataset ở nơi khác, có thể đặt trực tiếp `DATASET_ROOT = Path("...")` sau khi cell này chạy.

In [ ]:
try:
    import kagglehub
except ImportError as exc:
    raise ImportError(
        "Chưa có kagglehub. Hãy chạy: !pip install kagglehub"
    ) from exc

print(f"Đang tải dataset {DATASET_SLUG} từ Kaggle...")
download_path = kagglehub.dataset_download(DATASET_SLUG)
DATASET_ROOT = Path(download_path)

print("Dataset root:", DATASET_ROOT)
print("Các mục cấp 1:")
for item in sorted(DATASET_ROOT.iterdir()):
    if item.is_dir():
        print(" -", item.name)

## 2. Phát hiện generator và kiểm tra cấu trúc dataset

Notebook không hard-code tên generator. Generator được xác định từ các thư mục con có ít nhất một phần của cấu trúc `train/ai`, `train/nature`, `val/ai`, `val/nature`.

In [ ]:
def looks_like_generator_dir(path: Path) -> bool:
    return path.is_dir() and any(
        (path / split / label).exists()
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )


# Một số bản tải có thể bọc dataset trong một thư mục con.
# Cell này tìm root gần nhất chứa các thư mục generator.
candidate_roots = [DATASET_ROOT] + [p for p in DATASET_ROOT.iterdir() if p.is_dir()]
root_matches = []
for candidate in candidate_roots:
    generator_dirs = sorted([p for p in candidate.iterdir() if looks_like_generator_dir(p)])
    if generator_dirs:
        root_matches.append((candidate, generator_dirs))

if not root_matches:
    raise FileNotFoundError(
        "Không tìm thấy thư mục generator có cấu trúc train/ai, train/nature, val/ai, val/nature "
        f"bên trong {DATASET_ROOT}"
    )

DATASET_ROOT, generator_dirs = max(root_matches, key=lambda item: len(item[1]))
generators = [p.name for p in generator_dirs]

structure_records = []
missing_dirs = []

for generator_dir in generator_dirs:
    for split in EXPECTED_SPLITS:
        for label_name in EXPECTED_LABELS:
            folder = generator_dir / split / label_name
            exists = folder.exists() and folder.is_dir()
            image_count = (
                sum(1 for x in folder.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS)
                if exists
                else 0
            )
            structure_records.append({
                "generator": generator_dir.name,
                "split": split,
                "label_name": label_name,
                "folder": str(folder),
                "exists": exists,
                "image_count": image_count,
            })
            if not exists:
                missing_dirs.append(str(folder))

structure_df = pd.DataFrame(structure_records)

print("DATASET_ROOT được dùng:", DATASET_ROOT)
print("Số generator tìm thấy:", len(generators))
print("Generator tìm thấy:", generators)

if missing_dirs:
    print("\nCác thư mục còn thiếu:")
    for folder in missing_dirs:
        print(" -", folder)
else:
    print("\nCấu trúc train/val và ai/nature đầy đủ cho tất cả generator.")

display(structure_df)

## 3. Quét ảnh, kiểm tra bằng PIL và tính SHA-256

Cell này tạo metadata ở mức ảnh. Mỗi ảnh được mở bằng PIL để lấy kích thước và định dạng; ảnh lỗi được ghi nhận nhưng không đưa vào metadata hợp lệ.

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


valid_records = []
error_records = []

for generator_dir in tqdm(generator_dirs, desc="Generators"):
    for split in EXPECTED_SPLITS:
        for source_label_name in EXPECTED_LABELS:
            folder = generator_dir / split / source_label_name
            if not folder.exists():
                continue

            image_paths = sorted([
                p for p in folder.rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXTS
            ])
            for img_path in tqdm(
                image_paths,
                desc=f"{generator_dir.name}/{split}/{source_label_name}",
                leave=False,
            ):
                base_record = {
                    "image_path": str(img_path.resolve()),
                    "relative_path": img_path.relative_to(DATASET_ROOT).as_posix(),
                    "generator": generator_dir.name,
                    "original_split": split,
                    "label": LABEL_MAP[source_label_name],
                    "label_name": LABEL_DISPLAY[source_label_name],
                    "file_size": img_path.stat().st_size,
                }
                try:
                    with Image.open(img_path) as img:
                        img.verify()

                    with Image.open(img_path) as img:
                        width, height = img.size
                        image_format = img.format

                    valid_records.append({
                        **base_record,
                        "width": width,
                        "height": height,
                        "image_format": image_format,
                        "sha256": sha256_file(img_path),
                    })
                except (UnidentifiedImageError, OSError, ValueError) as exc:
                    error_records.append({
                        **base_record,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                    })

metadata_df = pd.DataFrame(valid_records)
error_df = pd.DataFrame(error_records)

required_columns = [
    "image_path", "relative_path", "generator", "original_split", "label", "label_name",
    "width", "height", "image_format", "file_size", "sha256",
]
if metadata_df.empty:
    metadata_df = pd.DataFrame(columns=required_columns)
else:
    metadata_df = metadata_df[required_columns].sort_values("relative_path").reset_index(drop=True)

print("Số ảnh hợp lệ:", len(metadata_df))
print("Số ảnh lỗi:", len(error_df))
display(metadata_df.head())
if not error_df.empty:
    display(error_df.head())

## 4. Phát hiện ảnh trùng và kiểm tra overlap SHA-256

Ảnh trùng được phát hiện bằng SHA-256 trên các ảnh hợp lệ. Overlap train/test trong TH1 cũng được kiểm tra bằng SHA-256 sau khi sinh metadata cross-generator.

In [ ]:
if metadata_df.empty:
    duplicate_df = pd.DataFrame(columns=list(metadata_df.columns) + ["duplicate_count"])
else:
    sha_counts = metadata_df["sha256"].value_counts()
    duplicate_hashes = sha_counts[sha_counts > 1]
    duplicate_df = metadata_df[metadata_df["sha256"].isin(duplicate_hashes.index)].copy()
    duplicate_df["duplicate_count"] = duplicate_df["sha256"].map(duplicate_hashes)
    duplicate_df = duplicate_df.sort_values(["sha256", "relative_path"]).reset_index(drop=True)

print("Số nhóm SHA-256 bị trùng:", duplicate_df["sha256"].nunique() if not duplicate_df.empty else 0)
print("Số ảnh nằm trong các nhóm trùng:", len(duplicate_df))
if not duplicate_df.empty:
    display(duplicate_df.head(20))

## 5. TH1 - Cross-generator metadata

Yêu cầu:

- `BASE_GENERATOR` phải tồn tại.
- Training metadata chỉ lấy `train` của `BASE_GENERATOR`.
- Test metadata lấy `val` của `BASE_GENERATOR` và `val` của tất cả generator còn lại.
- Không đưa fake của generator khác vào training metadata.
- Giữ nguyên split gốc qua cột `original_split`.

In [ ]:
# TH1: Cross-generator metadata cho từng generator
# Mỗi generator sẽ có một file train riêng:
#   train_base_generator_<generator>.csv
# Test vẫn dùng toàn bộ split val của tất cả generator.

cross_root = OUTPUT_ROOT / "cross_generator"
train_by_generator_root = cross_root / "train_by_generator"
test_by_generator_root = cross_root / "test_by_generator"
train_by_generator_root.mkdir(parents=True, exist_ok=True)
test_by_generator_root.mkdir(parents=True, exist_ok=True)

test_all_df = metadata_df[
    metadata_df["original_split"] == "val"
].copy().reset_index(drop=True)

if test_all_df.empty:
    raise ValueError(
        "Test metadata bị rỗng. Không tìm thấy ảnh val trong metadata_df."
    )

test_all_path = cross_root / "test_all_generators.csv"
test_all_df.to_csv(test_all_path, index=False)

test_by_generator_paths = []
for generator in sorted(test_all_df["generator"].unique()):
    generator_test_df = test_all_df[test_all_df["generator"] == generator].copy().reset_index(drop=True)
    out_path = test_by_generator_root / f"{generator}.csv"
    generator_test_df.to_csv(out_path, index=False)
    test_by_generator_paths.append(out_path)

train_by_generator_paths = []
overlap_records = []

for base_generator in sorted(generators):
    train_base_df = metadata_df[
        (metadata_df["generator"] == base_generator) &
        (metadata_df["original_split"] == "train")
    ].copy().reset_index(drop=True)

    if train_base_df.empty:
        raise ValueError(
            f"Training metadata bị rỗng. Không tìm thấy ảnh train cho generator={base_generator!r}."
        )

    other_fake_in_train = train_base_df[
        (train_base_df["generator"] != base_generator) &
        (train_base_df["label_name"] == "fake")
    ]
    if not other_fake_in_train.empty:
        raise AssertionError(
            f"Training metadata của {base_generator} chứa fake của generator khác, cần kiểm tra lại logic lọc."
        )

    train_base_path = train_by_generator_root / f"train_base_generator_{base_generator}.csv"
    train_base_df.to_csv(train_base_path, index=False)
    train_by_generator_paths.append(train_base_path)

    train_hashes = set(train_base_df["sha256"])
    test_hashes = set(test_all_df["sha256"])
    overlap_hashes_for_generator = train_hashes & test_hashes

    for sha256 in sorted(overlap_hashes_for_generator):
        overlap_records.append({
            "base_generator": base_generator,
            "sha256": sha256,
        })

overlap_summary_df = pd.DataFrame(overlap_records)
overlap_path = cross_root / "train_test_sha256_overlap.csv"
overlap_summary_df.to_csv(overlap_path, index=False)

# Giữ các biến này để các cell thống kê/report phía sau dùng được.
BASE_GENERATOR = "ALL_BASE_GENERATORS"
train_mode = "cross_generator_per_base"
train_base_path = train_by_generator_root
train_base_df = pd.concat(
    [pd.read_csv(path) for path in train_by_generator_paths],
    ignore_index=True,
)
overlap_hashes = set(overlap_summary_df["sha256"]) if not overlap_summary_df.empty else set()
overlap_df = overlap_summary_df

print("Đã lưu TH1 cross-generator metadata theo từng generator:")
print("- Train theo từng generator:", train_by_generator_root)
print("- Test tất cả generator:", test_all_path)
print("- Test theo từng generator:", test_by_generator_root)
print("- File overlap SHA-256:", overlap_path)
print("- Số file train riêng:", len(train_by_generator_paths))
print("- Số dòng test metadata:", len(test_all_df))
print("- Số SHA-256 overlap train/test:", len(overlap_hashes))

for path in train_by_generator_paths:
    print("  -", path)

if not overlap_df.empty:
    display(overlap_df.head(20))


## 6. TH2 - Combined metadata

Gộp toàn bộ ảnh hợp lệ của tất cả generator vào một metadata chung. Không chia lại dữ liệu, không tạo train/test mới, giữ nguyên `original_split`.

In [ ]:
combined_root = OUTPUT_ROOT / "combined"
combined_root.mkdir(parents=True, exist_ok=True)

combined_path = combined_root / "all_generators_metadata.csv"
metadata_df.to_csv(combined_path, index=False)

print("Đã lưu TH2 combined metadata:")
print(" -", combined_path)

## 7. Lưu file kiểm tra dữ liệu và thống kê

Các thống kê trong cell này được tính trực tiếp từ dữ liệu sau khi notebook chạy.

In [ ]:
checks_root = OUTPUT_ROOT / "checks"
checks_root.mkdir(parents=True, exist_ok=True)

error_path = checks_root / "invalid_images.csv"
duplicate_path = checks_root / "duplicate_images_sha256.csv"
structure_path = checks_root / "dataset_structure_check.csv"
stats_path = checks_root / "dataset_statistics.json"

error_df.to_csv(error_path, index=False)
duplicate_df.to_csv(duplicate_path, index=False)
structure_df.to_csv(structure_path, index=False)

if metadata_df.empty:
    label_stats = pd.DataFrame(columns=["label_name", "label", "count"])
    generator_stats = pd.DataFrame(columns=["generator", "count"])
    split_stats = pd.DataFrame(columns=["original_split", "count"])
    generator_split_label_stats = pd.DataFrame(
        columns=["generator", "original_split", "label_name", "label", "count"]
    )
else:
    label_stats = metadata_df.groupby(["label_name", "label"]).size().reset_index(name="count")
    generator_stats = metadata_df.groupby("generator").size().reset_index(name="count")
    split_stats = metadata_df.groupby("original_split").size().reset_index(name="count")
    generator_split_label_stats = (
        metadata_df.groupby(["generator", "original_split", "label_name", "label"])
        .size()
        .reset_index(name="count")
    )

train_test_overlap_count = len(overlap_hashes) if "overlap_hashes" in globals() else 0

# Các biến đường dẫn này thường được tạo ở cell TH1.
# Fallback giúp cell thống kê không bị NameError nếu bạn chạy lại từng cell.
cross_root = globals().get("cross_root", OUTPUT_ROOT / "cross_generator")
test_by_generator_root = globals().get("test_by_generator_root", cross_root / "test_by_generator")
train_base_path = globals().get("train_base_path", cross_root / "train_base_generator.csv")
test_all_path = globals().get("test_all_path", cross_root / "test_all_generators.csv")
overlap_path = globals().get("overlap_path", cross_root / "train_test_sha256_overlap.csv")
combined_path = globals().get("combined_path", OUTPUT_ROOT / "combined" / "all_generators_metadata.csv")

stats = {
    "dataset_root": str(DATASET_ROOT),
    "base_generator": BASE_GENERATOR,
    "num_generators": len(generators),
    "generators": generators,
    "num_valid_images": int(len(metadata_df)),
    "num_invalid_images": int(len(error_df)),
    "num_duplicate_sha256_groups": int(duplicate_df["sha256"].nunique()) if not duplicate_df.empty else 0,
    "num_duplicate_images": int(len(duplicate_df)),
    "num_train_test_overlap_sha256": int(train_test_overlap_count),
    "label_stats": label_stats.to_dict(orient="records"),
    "generator_stats": generator_stats.to_dict(orient="records"),
    "split_stats": split_stats.to_dict(orient="records"),
    "generator_split_label_stats": generator_split_label_stats.to_dict(orient="records"),
    "metadata_files": {
        "cross_generator_train": str(train_base_path),
        "cross_generator_test_all": str(test_all_path),
        "cross_generator_test_by_generator_dir": str(test_by_generator_root),
        "combined_all": str(combined_path),
        "invalid_images": str(error_path),
        "duplicate_images": str(duplicate_path),
        "train_test_overlap": str(overlap_path),
        "structure_check": str(structure_path),
    },
}

with stats_path.open("w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("Đã lưu file kiểm tra và thống kê:")
for path in [error_path, duplicate_path, overlap_path, structure_path, stats_path]:
    print(" -", path)

print("\nSố ảnh real/fake:")
display(label_stats)
print("\nSố ảnh theo generator:")
display(generator_stats)
print("\nSố ảnh theo split:")
display(split_stats)
print("\nSố ảnh theo generator/split/label:")
display(generator_split_label_stats)

## 8. Sinh báo cáo Tuần 2

Báo cáo được sinh sau khi notebook chạy và dùng thống kê thực tế từ metadata/checks. Nếu notebook chưa chạy, không có số liệu nào được khẳng định bên ngoài file report.

In [ ]:
def markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "Không có dữ liệu."
    columns = list(df.columns)
    rows = df.astype(str).values.tolist()
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, separator] + body)


REPORT_ROOT.mkdir(parents=True, exist_ok=True)
report_path = REPORT_ROOT / "week2_dataset_organization_report.md"

report = f"""# Báo cáo Tuần 2 - Tổ chức dataset và chuẩn bị metadata

## 1. Mục tiêu

Chuẩn bị metadata cho dataset Tiny-GenImage phục vụ các thí nghiệm phát hiện ảnh thật/ảnh AI-generated. Notebook chỉ thực hiện kiểm tra dữ liệu, quét ảnh, xác thực ảnh, phát hiện lỗi/trùng lặp và sinh metadata; không huấn luyện model.

## 2. Cấu trúc dataset

Dataset root được dùng sau khi chạy notebook:

```text
{DATASET_ROOT}
```

Cấu trúc mong đợi:

```text
Generator/
  train/
    ai/
    nature/
  val/
    ai/
    nature/
```

Kết quả kiểm tra cấu trúc:

{markdown_table(structure_df)}

## 3. Quy trình tổ chức dữ liệu

Notebook giữ nguyên dữ liệu gốc và không copy ảnh sang cấu trúc train/test mới. Tên generator được lấy từ tên thư mục generator thực tế. Split gốc `train` và `val` được giữ trong cột `original_split`.

## 4. Quy trình sinh metadata

Mỗi file ảnh có phần mở rộng hợp lệ được mở bằng PIL để xác thực. Ảnh hợp lệ được ghi metadata gồm đường dẫn, generator, split gốc, label, kích thước, định dạng, dung lượng file và SHA-256. Ảnh lỗi được ghi riêng vào file kiểm tra.

Các cột metadata chính:

```text
image_path, relative_path, generator, original_split, label, label_name,
width, height, image_format, file_size, sha256
```

Quy ước nhãn:

```text
real = 0
fake = 1
```

## 5. TH1: Cross-generator

`BASE_GENERATOR` được cấu hình là `{BASE_GENERATOR}`.

Training metadata chỉ lấy ảnh thuộc split `train` của `{BASE_GENERATOR}`. Test metadata lấy toàn bộ split `val` của tất cả generator, bao gồm `{BASE_GENERATOR}` và các generator còn lại. Notebook không đưa fake của generator khác vào training metadata.

File sinh ra:

```text
{train_base_path}
{test_all_path}
{test_by_generator_root}/<generator>.csv
```

Số dòng training metadata: {len(train_base_df)}

Số dòng test metadata: {len(test_all_df)}

Số SHA-256 overlap giữa training và test metadata: {train_test_overlap_count}

## 6. TH2: Combined metadata

Combined metadata gộp toàn bộ ảnh hợp lệ của tất cả generator. Notebook không chia lại dữ liệu và giữ nguyên `original_split`.

File sinh ra:

```text
{combined_path}
```

Số dòng combined metadata: {len(metadata_df)}

## 7. Kiểm tra dữ liệu

Các kiểm tra đã thực hiện:

- Kiểm tra cấu trúc thư mục generator/split/label.
- Kiểm tra ảnh hợp lệ bằng PIL.
- Ghi nhận ảnh lỗi.
- Phát hiện ảnh trùng bằng SHA-256.
- Kiểm tra overlap SHA-256 giữa training và test metadata trong TH1.
- Kiểm tra `BASE_GENERATOR` tồn tại trong danh sách generator thực tế.

File kiểm tra:

```text
{error_path}
{duplicate_path}
{overlap_path}
{structure_path}
{stats_path}
```

## 8. Thống kê

Số generator: {len(generators)}

Danh sách generator:

```text
{', '.join(generators)}
```

Số ảnh hợp lệ: {len(metadata_df)}

Số ảnh lỗi: {len(error_df)}

Số nhóm SHA-256 bị trùng: {duplicate_df['sha256'].nunique() if not duplicate_df.empty else 0}

Số ảnh nằm trong các nhóm trùng: {len(duplicate_df)}

### Số ảnh real/fake

{markdown_table(label_stats)}

### Số ảnh theo generator

{markdown_table(generator_stats)}

### Số ảnh theo split

{markdown_table(split_stats)}

### Số ảnh theo generator/split/label

{markdown_table(generator_split_label_stats)}

## 9. Kết quả

Notebook đã sinh metadata và các file kiểm tra trong `data/metadata/`, đồng thời sinh báo cáo này trong `report/` dựa trên dữ liệu thực tế sau khi chạy.

## 10. Kết luận

Dữ liệu đã được tổ chức ở mức metadata cho hai trường hợp thí nghiệm: cross-generator và combined metadata. Các bước tiếp theo có thể dùng trực tiếp các file CSV đã sinh để xây dataset loader hoặc chạy thí nghiệm baseline ở tuần sau.
"""

report_path.write_text(report, encoding="utf-8")
print("Đã sinh báo cáo:", report_path)

In [ ]:
# Lưu dữ liệu đã chuẩn bị sang Google Drive để tái sử dụng ở notebook khác.
# Chạy cell này sau khi đã sinh xong metadata và report.

from pathlib import Path
import shutil

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Cell này cần chạy trên Google Colab để mount Google Drive.") from exc

drive.mount("/content/drive")

DRIVE_PREPARED_ROOT = Path("/content/drive/MyDrive/genimage_prepared_data")
DRIVE_PREPARED_ROOT.mkdir(parents=True, exist_ok=True)

copy_targets = {
    "metadata": OUTPUT_ROOT,
    "report": REPORT_ROOT,
}

for name, source_dir in copy_targets.items():
    if not source_dir.exists():
        raise FileNotFoundError(
            f"Chưa tìm thấy {source_dir}. Hãy chạy các cell sinh metadata/report trước."
        )

    target_dir = DRIVE_PREPARED_ROOT / name
    shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
    print(f"Đã lưu {name}: {target_dir}")

print("Hoàn tất lưu dữ liệu đã chuẩn bị vào Google Drive:", DRIVE_PREPARED_ROOT)


## 9. Summary

Cell cuối in tóm tắt các artefact đã sinh sau khi notebook chạy.

In [ ]:
metadata_outputs = [
    train_base_path,
    test_all_path,
    combined_path,
    error_path,
    duplicate_path,
    overlap_path,
    structure_path,
    stats_path,
]
metadata_outputs.extend(test_by_generator_paths)

summary = {
    "num_generators": len(generators),
    "num_valid_images": len(metadata_df),
    "num_invalid_images": len(error_df),
    "num_duplicate_sha256_groups": duplicate_df["sha256"].nunique() if not duplicate_df.empty else 0,
    "num_duplicate_images": len(duplicate_df),
    "metadata_outputs": [str(path) for path in metadata_outputs],
    "report_output": str(report_path),
}

print("SUMMARY")
print("- Số generator:", summary["num_generators"])
print("- Số ảnh hợp lệ:", summary["num_valid_images"])
print("- Số ảnh lỗi:", summary["num_invalid_images"])
print("- Số nhóm SHA-256 bị trùng:", summary["num_duplicate_sha256_groups"])
print("- Số ảnh nằm trong các nhóm trùng:", summary["num_duplicate_images"])
print("- Metadata đã sinh:")
for path in summary["metadata_outputs"]:
    print("  -", path)
print("- Report đã sinh:", summary["report_output"])